In [33]:
import numpy as np
import pandas as pd

from astropy.io import fits
from astropy.wcs import WCS
from astropy.convolution import Gaussian2DKernel, convolve

from reproject import reproject_interp

import pyregion



# FILES

muse_file = (
    "/Users/leianydeoleo/Downloads/"
    "Hbeta+Halpha IIZW096 MUSE.fits"
)

sitelle_file = (
    "/Users/leianydeoleo/IIZw96_SN3/MAPS/"
    "IIZw96_SN3.LineMaps.map.6563.1x1.flux.fits"
)

sitelle_error_file = (
    "/Users/leianydeoleo/IIZw96_SN3/MAPS/"
    "IIZw96_SN3.LineMaps.map.6563.1x1.flux-err.fits"
)

region_file = (
    "/Users/leianydeoleo/Desktop/IIZW96/"
    "galaxy.reg"
)



# LOAD MUSE H-ALPHA FLUX

muse_hdul = fits.open(
    muse_file
)

muse_flux = (
    muse_hdul["M0_HALPHA"]
    .data
    .astype(float)
)

muse_error = (
    muse_hdul["ERR_HALPHA"]
    .data
    .astype(float)
)

muse_header = (
    muse_hdul["M0_HALPHA"]
    .header
)

muse_wcs = WCS(
    muse_header
)



# LOAD SITELLE H-ALPHA FLUX


sitelle_hdul = fits.open(
    sitelle_file
)

sitelle_flux = (
    sitelle_hdul[0]
    .data
    .astype(float)
)

sitelle_header = (
    sitelle_hdul[0]
    .header
)

sitelle_wcs = WCS(
    sitelle_header
)



# LOAD SITELLE H-ALPHA ERROR

sitelle_error = fits.getdata(
    sitelle_error_file
).astype(float)



# PRINT IMAGE SHAPES

print(
    "MUSE original shape:",
    muse_flux.shape
)

print(
    "SITELLE shape:",
    sitelle_flux.shape
)



# REPROJECT MUSE ONTO SITELLE GRID

muse_reproj, footprint = reproject_interp(
    (muse_flux, muse_wcs),
    sitelle_wcs,
    shape_out=sitelle_flux.shape
)


# Reproject MUSE error map using same transformation

muse_error_reproj, footprint_error = (
    reproject_interp(
        (muse_error, muse_wcs),
        sitelle_wcs,
        shape_out=sitelle_flux.shape
    )
)


print(
    "Reprojected MUSE shape:",
    muse_reproj.shape
)



# MATCH SPATIAL RESOLUTION

# MUSE WFM seeing range
muse_fwhm = (
    0.76 + 0.94
) / 2


# SITELLE measured seeing
sitelle_fwhm = 1.08


# SITELLE pixel scale
sitelle_pixscale = 0.32


print(
    "MUSE FWHM:",
    muse_fwhm,
    "arcsec"
)

print(
    "SITELLE FWHM:",
    sitelle_fwhm,
    "arcsec"
)



# CONVOLVE MUSE TO MATCH SITELLE PSF

if sitelle_fwhm > muse_fwhm:

    # Required convolution FWHM
    sigma_arcsec = (
        np.sqrt(
            sitelle_fwhm**2
            -
            muse_fwhm**2
        )
        /
        2.355
    )


    # Convert sigma from arcsec to SITELLE pixels
    sigma_pix = (
        sigma_arcsec
        /
        sitelle_pixscale
    )


    print(
        "Convolution sigma:",
        sigma_arcsec,
        "arcsec"
    )

    print(
        "Convolution sigma:",
        sigma_pix,
        "pixels"
    )


    # Create Gaussian convolution kernel
    kernel = Gaussian2DKernel(
        sigma_pix
    )


    # Convolve MUSE flux
    muse_match = convolve(
        muse_reproj,
        kernel,
        preserve_nan=True
    )


    # Convolve MUSE error map
    muse_error_match = convolve(
        muse_error_reproj,
        kernel,
        preserve_nan=True
    )


else:

    # No additional convolution needed
    muse_match = muse_reproj

    muse_error_match = (
        muse_error_reproj
    )



# LOAD DS9 REGION USING PYREGION

print(
    "\nLoading DS9 aperture..."
)


reg = pyregion.open(
    region_file
)


print(
    "Number of regions found:",
    len(reg)
)


# CONVERT REGION TO IMAGE COORDINATES
# Convert DS9 region to SITELLE pixel coordinates

# The SITELLE WCS is used to transform the region
# into the coordinate system of the SITELLE image.

reg_image = reg.as_imagecoord(
    sitelle_wcs.to_header()
)


print(
    "Region converted to SITELLE image coordinates."
)


# CREATE APERTURE MASK


# Use the SITELLE image dimensions to create the region mask.

aperture_mask = reg_image.get_mask(
    shape=sitelle_flux.shape
)


# Convert to Boolean
aperture_mask = (
    aperture_mask.astype(bool)
)


print(
    "Aperture pixels:",
    np.sum(aperture_mask)
)


# CREATE COMMON VALID-DATA MASK


# The final mask requires:
#
# 1. Pixel inside DS9 galaxy aperture
# 2. Valid MUSE flux
# 3. Valid MUSE error
# 4. Valid SITELLE flux
# 5. Valid SITELLE error
# 6. Pixel covered by reprojected MUSE data

mask = (
    aperture_mask
    &
    np.isfinite(muse_match)
    &
    np.isfinite(muse_error_match)
    &
    np.isfinite(sitelle_flux)
    &
    np.isfinite(sitelle_error)
    &
    (footprint > 0)
)


print(
    "Pixels used in final comparison:",
    np.sum(mask)
)



# TOTAL MUSE FLUX

muse_total_flux = np.nansum(
    muse_match[mask]
)



# TOTAL SITELLE FLUX
sitelle_total_flux = np.nansum(
    sitelle_flux[mask]
)



# TOTAL MUSE ERROR

muse_total_error = np.sqrt(
    np.nansum(
        muse_error_match[mask]**2
    )
)


# TOTAL SITELLE ERROR

sitelle_total_error = np.sqrt(
    np.nansum(
        sitelle_error[mask]**2
    )
)



# FLUX RATIO

flux_ratio = (
    muse_total_flux
    /
    sitelle_total_flux
)



# ERROR ON FLUX RATIO

ratio_error = (
    flux_ratio
    *
    np.sqrt(
        (
            muse_total_error
            /
            muse_total_flux
        )**2
        +
        (
            sitelle_total_error
            /
            sitelle_total_flux
        )**2
    )
)



# PERCENT DIFFERENCE

percent_difference = (
    (
        muse_total_flux
        -
        sitelle_total_flux
    )
    /
    sitelle_total_flux
) * 100



# PRINT RESULTS

print(
    "\n======================================"
)

print(
    "MUSE + SITELLE FLUX COMPARISON"
)

print(
    "======================================"
)


print(
    f"MUSE total flux = "
    f"{muse_total_flux:.4e}"
)


print(
    f"MUSE total error = "
    f"{muse_total_error:.4e}"
)


print(
    f"SITELLE total flux = "
    f"{sitelle_total_flux:.4e}"
)


print(
    f"SITELLE total error = "
    f"{sitelle_total_error:.4e}"
)


print(
    f"MUSE/SITELLE ratio = "
    f"{flux_ratio:.4f} "
    f"+/- "
    f"{ratio_error:.4f}"
)


print(
    f"Percent difference = "
    f"{percent_difference:.2f}%"
)


print(
    "======================================"
)

MUSE original shape: (318, 317)
SITELLE shape: (2064, 2048)
Reprojected MUSE shape: (2064, 2048)
MUSE FWHM: 0.85 arcsec
SITELLE FWHM: 1.08 arcsec
Convolution sigma: 0.28291219032042336 arcsec
Convolution sigma: 0.884100594751323 pixels

Loading DS9 aperture...
Number of regions found: 1
Region converted to SITELLE image coordinates.
Aperture pixels: 8657
Pixels used in final comparison: 7368

MUSE + SITELLE FLUX COMPARISON
MUSE total flux = 3.2449e-13
MUSE total error = 7.7818e-17
SITELLE total flux = 7.0084e-13
SITELLE total error = 2.2050e-15
MUSE/SITELLE ratio = 0.4630 +/- 0.0015
Percent difference = -53.70%


In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from astropy.convolution import Gaussian2DKernel, convolve
from reproject import reproject_interp


# FILES

muse_file = "/Users/leianydeoleo/Downloads/Hbeta+Halpha IIZW096 MUSE.fits"

sitelle_file = (
    "/Users/leianydeoleo/IIZw96_SN3/MAPS/IIZw96_SN3.LineMaps.map.6563.1x1.flux.fits"
)


# LOAD MUSE

muse_hdul = fits.open(muse_file)

muse_flux = muse_hdul["M0_HALPHA"].data.astype(float)

muse_error = muse_hdul["ERR_HALPHA"].data.astype(float)

muse_header = muse_hdul["M0_HALPHA"].header
muse_wcs = WCS(muse_header)


# LOAD SITELLE

sitelle_hdul = fits.open(sitelle_file)

sitelle_flux = sitelle_hdul[0].data.astype(float)

sitelle_header = sitelle_hdul[0].header
sitelle_wcs = WCS(sitelle_header)


# LOAD SITELLE ERROR MAP
# CHANGE EXTENSION IF NEEDED

sitelle_error_file = '/Users/leianydeoleo/IIZw96_SN3/MAPS/IIZw96_SN3.LineMaps.map.6563.1x1.flux-err.fits'

sitelle_error = fits.getdata(
    sitelle_error_file
).astype(float)

fwhm_ave = (0.76+0.94)/2 #muse fwhm range 

muse_fwhm = fwhm_ave 
sitelle_fwhm = 1.08

sitelle_pixscale = 0.32

# ============================================================
# LOAD DS9 REGION
# ============================================================

import pyregion

region_file = (
    "/Users/leianydeoleo/Desktop/IIZW96/galaxy.reg"
)


# ============================================================
# CONVERT DS9 REGION TO SITELLE IMAGE COORDINATES
# ============================================================

reg = pyregion.open(
    region_file
).as_imagecoord(
    sitelle_wcs.to_header()
)


# ============================================================
# CREATE APERTURE MASK
# ============================================================

region_mask = reg.get_mask(
    hdu=sitelle_hdul[0]
)


aperture_mask = (
    region_mask.astype(bool)
)


print(
    "Number of pixels inside aperture:",
    np.sum(aperture_mask)
)

if sitelle_fwhm > muse_fwhm:

    sigma_arcsec = np.sqrt(
        sitelle_fwhm**2 - muse_fwhm**2
    ) / 2.355

    sigma_pix = sigma_arcsec / sitelle_pixscale

    kernel = Gaussian2DKernel(sigma_pix)

    muse_match = convolve(
        muse_reproj,
        kernel,
        preserve_nan=True
    )

    muse_error_match = convolve(
        muse_error_reproj,
        kernel,
        preserve_nan=True
    )

else:

    muse_match = muse_reproj
    muse_error_match = muse_error_reproj


# CREATE PIXEL GRID
ny, nx = sitelle_flux.shape

y, x = np.indices(
    (ny, nx)
)



# CREATE CIRCULAR APERTURE
aperture_mask = (
    (x - x0)**2
    +
    (y - y0)**2
    <= radius_pix**2
)



# COMMON VALID-DATA MASK
mask = (
    aperture_mask
    &
    np.isfinite(muse_match)
    &
    np.isfinite(muse_error_match)
    &
    np.isfinite(sitelle_flux)
    &
    np.isfinite(sitelle_error)
    &
    (footprint > 0)
)

# TOTAL FLUXES
muse_total_flux = np.nansum(
    muse_match[mask]
)

sitelle_total_flux = np.nansum(
    sitelle_flux[mask]
)


# TOTAL ERRORS
muse_total_error = np.sqrt(
    np.nansum(
        muse_error_match[mask]**2
    )
)

sitelle_total_error = np.sqrt(
    np.nansum(
        sitelle_error[mask]**2
    )
)



# FLUX RATIO

flux_ratio = (
    muse_total_flux /
    sitelle_total_flux
)

ratio_error = flux_ratio * np.sqrt(
    (muse_total_error / muse_total_flux)**2
    +
    (sitelle_total_error / sitelle_total_flux)**2
)


percent_difference = (
    (muse_total_flux - sitelle_total_flux)
    /
    sitelle_total_flux
) * 100


print(
    f"MUSE total flux = "
    f"{muse_total_flux:.4e}"
)

print(
    f"SITELLE total flux = "
    f"{sitelle_total_flux:.4e}"
)

print(
    f"MUSE/SITELLE ratio = "
    f"{flux_ratio:.4f}"
)

print(
    f"Ratio uncertainty = "
    f"{ratio_error:.4f}"
)

print(
    f"Percent difference = "
    f"{percent_difference:.2f}%"
)

In [16]:
ratio_galaxy_reprojected = sitelle_total_flux/muse_total_flux
print(f"SITELLE/MUSE total flux ratio on a reprojected pixel grid = "
    f"{ratio_galaxy_reprojected}")

SITELLE/MUSE total flux ratio on a reprojected pixel grid = 2.1592178814909007


In [19]:
import numpy as np
from astropy.io import fits
import re
import pandas as pd


# Input files

region_file = '/Users/leianydeoleo/Desktop/IIZW96/muse_galaxy.reg'
muse_file = "/Users/leianydeoleo/Downloads/Hbeta+Halpha IIZW096 MUSE.fits"



# Load MUSE Halpha maps

hdul = fits.open(muse_file)

flux = hdul["M0_HALPHA"].data
flux_err = hdul["ERR_HALPHA"].data

hdul.close()

print("Flux shape:", flux.shape)
print("Error shape:", flux_err.shape)


# Read DS9 IMAGE region
# Example:
# image
# circle(150,120,20)

with open(region_file, "r") as f:
    text = f.read()

match = re.search(r"circle\((.*?)\)", text)

if match is None:
    raise ValueError("No DS9 circle region found")

values = match.group(1).split(",")

x0 = float(values[0])
y0 = float(values[1])
radius = float(values[2].replace('"',''))



# Create circular mask

yy, xx = np.indices(flux.shape)

mask = (
    (xx - x0)**2 +
    (yy - y0)**2
    <= radius**2
)



# Select valid pixels

good = (
    mask &
    np.isfinite(flux) &
    np.isfinite(flux_err)
)


print("Pixels inside region:", np.sum(mask))
print("Valid pixels:", np.sum(good))



# Integrated flux

total_MUSE_flux = np.sum(flux[good])



# Flux uncertainty

total_MUSE_error = np.sqrt(
    np.sum(flux_err[good]**2)
)



# Output

print("--------------------------------")
print("MUSE Halpha integrated flux")
print("--------------------------------")
print(f"Flux  = {total_MUSE_flux:.5e} erg cm^-2 s^-1")
print(f"Error = {total_MUSE_error:.5e} erg cm^-2 s^-1")

if total_error > 0:
    print(f"S/N   = {total_MUSE_flux/total_MUSE_error:.2f}")

print("--------------------------------")

results = pd.DataFrame({
    "Instrument": ["MUSE"],
    "Line": ["Halpha"],
    "Flux_erg_cm2_s": [total_MUSE_flux],
    "Flux_Error_erg_cm2_s": [total_MUSE_error],
    "SNR": [total_MUSE_flux / total_MUSE_error],
    "N_pixels": [np.sum(good)]
})

results.to_csv("MUSE_Halpha_flux_measurement.csv", index=False)

print("Saved: MUSE_Halpha_flux_measurement.csv")

Flux shape: (318, 317)
Error shape: (318, 317)
Pixels inside region: 15947
Valid pixels: 15222
--------------------------------
MUSE Halpha integrated flux
--------------------------------
Flux  = 8.27827e-13 erg cm^-2 s^-1
Error = 1.29433e-16 erg cm^-2 s^-1
S/N   = 6395.80
--------------------------------
Saved: MUSE_Halpha_flux_measurement.csv


In [21]:
import numpy as np
from astropy.io import fits
import re

region_file = "/Users/leianydeoleo/Desktop/IIZW96/galaxy.reg" 
sitelle_file = (
    "/Users/leianydeoleo/IIZw96_SN3/MAPS/IIZw96_SN3.LineMaps.map.6563.1x1.flux.fits"
)

sitelle_error_file = '/Users/leianydeoleo/IIZw96_SN3/MAPS/IIZw96_SN3.LineMaps.map.6563.1x1.flux-err.fits'


# Load maps

flux = fits.getdata(sitelle_file)
flux_err = fits.getdata(sitelle_error_file)


# Read DS9 image region
# Example:
# image
# circle(1034,1348,20)

with open(region_file, "r") as f:
    text = f.read()

match = re.search(r"circle\((.*?)\)", text)

if match is None:
    raise ValueError("No circle region found")

values = match.group(1).split(",")

x0 = float(values[0])
y0 = float(values[1])
radius_pix = float(values[2].replace('"',''))


# Create mask
yy, xx = np.indices(flux.shape)

mask = (
    (xx - x0)**2 +
    (yy - y0)**2
    <= radius_pix**2
)


# Valid pixels
good = (
    mask &
    np.isfinite(flux) &
    np.isfinite(flux_err)
)

print("Image shape:", flux.shape)
print("Pixels inside region:", np.sum(mask))
print("Valid pixels:", np.sum(good))


# Integrated flux
total_flux = np.sum(flux[good])


# Integrated uncertainty
total_error = np.sqrt(np.sum(flux_err[good]**2))


# Results
print("--------------------------------")
print("Integrated emission-line flux")
print("--------------------------------")
print(f"Flux  = {total_flux:.4e} erg cm^-2 s^-1")
print(f"Error = {total_error:.4e} erg cm^-2 s^-1")

if total_error > 0:
    print(f"S/N   = {total_flux/total_error:.2f}")
else:
    print("S/N   = undefined")

print("--------------------------------")

results = pd.DataFrame({
    "Instrument": ["SITELLE"],
    "Line": ["Halpha"],
    "Flux_erg_cm2_s": [total_flux],
    "Flux_Error_erg_cm2_s": [total_error],
    "SNR": [total_flux / total_error],
    "N_pixels": [np.sum(good)]
})

results.to_csv("SITELLE_Halpha_flux_measurement.csv", index=False)

print("Saved: SITELLE_Halpha_flux_measurement.csv")

Image shape: (2064, 2048)
Pixels inside region: 8657
Valid pixels: 8657
--------------------------------
Integrated emission-line flux
--------------------------------
Flux  = 7.0617e-13 erg cm^-2 s^-1
Error = 2.2049e-15 erg cm^-2 s^-1
S/N   = 320.27
--------------------------------
Saved: SITELLE_Halpha_flux_measurement.csv


In [25]:
ratio_galaxy = total_flux/total_MUSE_flux
print(f"SITELLE/MUSE total flux ratio with an aperture adjusted to pixel grid = "
    f"{ratio_galaxy}")

SITELLE/MUSE total flux ratio with an aperture adjusted to pixel grid = 0.8530402886575532


In [24]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u



# SOURCE POSITIONS

sources = {

    "ID7": SkyCoord(
        314.351423,
        17.127521,
        unit="deg"
    ),

    "ID8": SkyCoord(
        314.351552,
        17.127566,
        unit="deg"
    )

}



# LOAD MUSE FLUX + ERROR

muse = fits.open(
    "/Users/leianydeoleo/Downloads/Hbeta+Halpha IIZW096 MUSE.fits"
)

muse_flux = muse["M0_HALPHA"].data.astype(float)

muse_error = muse["ERR_HALPHA"].data.astype(float)

muse_wcs = WCS(
    muse["M0_HALPHA"].header
)



# LOAD SITELLE FLUX + ERROR

sitelle = fits.open(
    "/Users/leianydeoleo/IIZw96_SN3/MAPS/IIZw96_SN3.LineMaps.map.6563.1x1.flux.fits"
)


sitelle_flux = sitelle[0].data.astype(float)

sitelle_wcs = WCS(
    sitelle[0].header
)


# If you have a separate SITELLE error file:
#
sitelle_error = fits.getdata(
    "/Users/leianydeoleo/IIZw96_SN3/MAPS/IIZw96_SN3.LineMaps.map.6563.1x1.flux-err.fits"
 ).astype(float)
#
# For now:
#sitelle_error = sitelle[0].data.astype(float)




# APERTURE

radius_arcsec = 1.0



# FUNCTION TO MEASURE FLUX AND ERROR

def measure_source(
    flux,
    error,
    wcs,
    coord,
    radius_arcsec
):

    # Sky -> pixel

    x, y = wcs.world_to_pixel(coord)


    # pixel scale

    pixscale = np.mean(
        np.abs(
            wcs.pixel_scale_matrix.diagonal()
        )
    ) * 3600


    radius_pix = radius_arcsec / pixscale


    # mask

    yy, xx = np.indices(
        flux.shape
    )


    mask = (
        (xx-x)**2 +
        (yy-y)**2
        <= radius_pix**2
    )


    good = (
        mask
        &
        np.isfinite(flux)
        &
        np.isfinite(error)
    )


    # integrated flux

    total_flux = np.sum(
        flux[good]
    )


    # uncertainty propagation

    total_error = np.sqrt(
        np.sum(
            error[good]**2
        )
    )


    snr = total_flux / total_error


    return (
        x,
        y,
        total_flux,
        total_error,
        snr,
        np.sum(good),
        radius_pix
    )



# RUN MEASUREMENTS

results = []


for name, coord in sources.items():

    for instrument, flux, error, wcs in [

        (
            "MUSE",
            muse_flux,
            muse_error,
            muse_wcs
        ),

        (
            "SITELLE",
            sitelle_flux,
            sitelle_error,
            sitelle_wcs
        )

    ]:


        (
            x,
            y,
            flux_total,
            error_total,
            snr,
            npix,
            radius_pix
        ) = measure_source(
            flux,
            error,
            wcs,
            coord,
            radius_arcsec
        )


        results.append({

            "Source": name,

            "Instrument": instrument,

            "RA_deg": coord.ra.deg,

            "Dec_deg": coord.dec.deg,

            "Pixel_x": x,

            "Pixel_y": y,

            "Radius_arcsec": radius_arcsec,

            "Radius_pixels": radius_pix,

            "Flux_erg_cm2_s": flux_total,

            "Flux_Error_erg_cm2_s": error_total,

            "SNR": snr,

            "N_pixels": npix

        })



# SAVE TABLE


table = pd.DataFrame(results)


table.to_csv(
    "MUSE_SITELLE_source_flux_errors.csv",
    index=False
)


print(table)

print(
    "\nSaved: MUSE_SITELLE_source_flux_errors.csv"
)

  Source Instrument      RA_deg    Dec_deg             Pixel_x  \
0    ID7       MUSE  314.351423  17.127521  164.49935208487716   
1    ID7    SITELLE  314.351423  17.127521   1054.407499137397   
2    ID8       MUSE  314.351552  17.127566  162.28032771885097   
3    ID8    SITELLE  314.351552  17.127566  1053.0404285421484   

              Pixel_y  Radius_arcsec  Radius_pixels  Flux_erg_cm2_s  \
0  124.04841741114095            1.0       5.000000    2.828125e-14   
1  1032.2932010386853            1.0       3.092832    2.298638e-14   
2  124.85841340968398            1.0       5.000000    2.664240e-14   
3   1032.773886328153            1.0       3.092832    2.356058e-14   

   Flux_Error_erg_cm2_s          SNR  N_pixels  
0          1.641377e-17  1723.019105        80  
1          8.759427e-17   262.418799        30  
2          1.613281e-17  1651.441997        78  
3          8.696584e-17   270.917679        31  

Saved: MUSE_SITELLE_source_flux_errors.csv


In [28]:
ratio_sources = 2.2986383102363E-14/2.82812478935556E-14
print(ratio_sources)

0.8127782475822387


In [29]:
ratio_sources1 = 2.356058e-14/2.664240e-14 
print(ratio_sources1)

0.8843264871032639
